# AI Image Detection — Full Dataset Training

Trains EfficientNet-B0 and DeiT-Tiny on **all ArtiFact generators**.

- Holds out 500 images per generator to Drive before training (for later testing)
- Trains on the remainder of every generator
- Saves best checkpoints and ONNX exports to Drive

## 1. Install dependencies

In [ ]:
!pip install -q timm==1.0.11 kagglehub pandas scikit-learn pillow onnxruntime onnxscript

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR     = '/content/drive/MyDrive/ai_detection_ckpts_full'
TEST_IMG_DIR = '/content/drive/MyDrive/ai_detection_test_images_full'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(TEST_IMG_DIR, exist_ok=True)
print(f'Checkpoints: {CKPT_DIR}')
print(f'Test images: {TEST_IMG_DIR}')

## 3. Kaggle credentials & dataset location

In [ ]:
import os, json as _json
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

with open('/root/.kaggle/kaggle.json') as f:
    creds = _json.load(f)
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY']      = creds['key']

import kagglehub
CACHE_PATH = os.path.expanduser('~/.cache/kagglehub/datasets/awsaf49/artifact-dataset/versions/1')
if os.path.isdir(CACHE_PATH):
    DATA_ROOT = CACHE_PATH
    print(f'Using cached dataset at: {DATA_ROOT}')
else:
    DATA_ROOT = kagglehub.dataset_download('awsaf49/artifact-dataset')
    print(f'Downloaded dataset to: {DATA_ROOT}')

## 4. Configuration

In [ ]:
import torch

IMG_COL    = 'image_path'
LABEL_COL  = 'target'
IMG_SIZE   = 224
BATCH_SIZE = 64
NUM_EPOCHS = 20
LR         = 1e-4
PATIENCE   = 6
VAL_SPLIT  = 0.1
NUM_WORKERS = 2
SEED       = 42
HOLDOUT_PER_GENERATOR = 500   # images per generator reserved for testing, never seen during training
MAX_IMAGES_PER_GENERATOR = 10000  # cap per generator for training (None = use all)

ALL_GENERATORS = [
    g for g in sorted(os.listdir(DATA_ROOT))
    if os.path.isdir(os.path.join(DATA_ROOT, g))
    and os.path.exists(os.path.join(DATA_ROOT, g, 'metadata.csv'))
]
print(f'All generators ({len(ALL_GENERATORS)}): {ALL_GENERATORS}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 5. Reserve holdout images to Drive

Splits off 500 images per generator BEFORE training. Resume-safe.

In [ ]:
import random, shutil, pandas as pd

random.seed(SEED)

MANIFEST_PATH = os.path.join(TEST_IMG_DIR, 'manifest.csv')

if os.path.exists(MANIFEST_PATH):
    print(f'Holdout manifest already exists, loading: {MANIFEST_PATH}')
    holdout_manifest = pd.read_csv(MANIFEST_PATH)
    holdout_indices = {gen: set(holdout_manifest[holdout_manifest.generator == gen]['orig_index'].tolist())
                       for gen in ALL_GENERATORS}
else:
    rows = []
    holdout_indices = {}
    for gen in ALL_GENERATORS:
        meta_path = os.path.join(DATA_ROOT, gen, 'metadata.csv')
        df = pd.read_csv(meta_path).reset_index(drop=True)
        n_holdout = min(HOLDOUT_PER_GENERATOR, len(df))
        held = df.sample(n=n_holdout, random_state=SEED)
        holdout_indices[gen] = set(held.index.tolist())

        gen_out = os.path.join(TEST_IMG_DIR, gen)
        os.makedirs(gen_out, exist_ok=True)
        for idx, row in held.iterrows():
            src = os.path.join(DATA_ROOT, gen, str(row[IMG_COL]))
            dst = os.path.join(gen_out, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            binary_label = 0 if int(row[LABEL_COL]) == 0 else 1
            rows.append({'generator': gen, 'drive_path': dst, 'label': binary_label, 'orig_index': idx})
        print(f'  {gen:30s}  {n_holdout} images held out')

    holdout_manifest = pd.DataFrame(rows)
    holdout_manifest.to_csv(MANIFEST_PATH, index=False)
    print(f'\nManifest saved → {MANIFEST_PATH}  ({len(holdout_manifest):,} total holdout images)')

## 6. Dataset class & transforms

In [ ]:
import random as _random
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

_random.seed(SEED)
torch.manual_seed(SEED)

class ArtiFactDataset(Dataset):
    def __init__(self, data_root, generators, transform=None, max_per_gen=None, exclude_indices=None):
        self.transform = transform
        self.samples = []
        exclude_indices = exclude_indices or {}

        for gen in generators:
            gen_dir   = os.path.join(data_root, gen)
            meta_path = os.path.join(gen_dir, 'metadata.csv')
            if not os.path.isdir(gen_dir) or not os.path.exists(meta_path):
                print(f'[WARN] skipping {gen}')
                continue

            df = pd.read_csv(meta_path).reset_index(drop=True)
            excl = exclude_indices.get(gen, set())
            df = df[~df.index.isin(excl)]

            if max_per_gen is not None and len(df) > max_per_gen:
                df = df.sample(n=max_per_gen, random_state=SEED).reset_index(drop=True)

            for _, row in df.iterrows():
                img_path = os.path.join(gen_dir, str(row[IMG_COL]))
                binary_label = 0 if int(row[LABEL_COL]) == 0 else 1
                self.samples.append((img_path, binary_label, gen))

        print(f'Loaded {len(self.samples):,} samples from {len(generators)} generators.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, gen = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, label, gen

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## 7. Build train / val loaders

In [ ]:
full_train = ArtiFactDataset(
    DATA_ROOT, ALL_GENERATORS,
    transform=train_tf,
    max_per_gen=MAX_IMAGES_PER_GENERATOR,
    exclude_indices=holdout_indices,
)

val_size   = int(len(full_train) * VAL_SPLIT)
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
val_ds.dataset = ArtiFactDataset(
    DATA_ROOT, ALL_GENERATORS,
    transform=eval_tf,
    max_per_gen=MAX_IMAGES_PER_GENERATOR,
    exclude_indices=holdout_indices,
)

def collate(batch):
    imgs, labels, gens = zip(*batch)
    return torch.stack(imgs), torch.tensor(labels), list(gens)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

print(f'Train: {len(train_ds):,}')
print(f'Val:   {len(val_ds):,}')

## 8. Model factory

In [ ]:
import timm
import torch.nn as nn

MODELS = {
    'efficientnet_b0': 'efficientnet_b0',
    'deit_tiny':       'deit_tiny_patch16_224',
}

def build_model(name):
    return timm.create_model(MODELS[name], pretrained=True, num_classes=2).to(DEVICE)

## 9. Train / eval loops

In [ ]:
import time, copy
from sklearn.metrics import accuracy_score

def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train_mode):
            logits = model(imgs)
            loss = criterion(logits, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / n, accuracy_score(all_labels, all_preds)


def train_model(name):
    print(f'\n{"="*60}\nTRAINING: {name}\n{"="*60}')
    model     = build_model(name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_acc  = 0.0
    best_val_loss = float('inf')
    best_state    = copy.deepcopy(model.state_dict())
    best_epoch    = 0
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, None)
        scheduler.step()

        print(f'[{name}] epoch {epoch:02d}/{NUM_EPOCHS}  '
              f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  '
              f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}  '
              f'({time.time()-t0:.0f}s)')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = copy.deepcopy(model.state_dict())
            best_epoch   = epoch
            ckpt_path = os.path.join(CKPT_DIR, f'{name}_best.pt')
            torch.save({
                'model_state_dict': best_state,
                'epoch': epoch,
                'val_loss': vl_loss,
                'val_acc':  vl_acc,
                'model_name': MODELS[name],
                'img_size': IMG_SIZE,
                'num_classes': 2,
            }, ckpt_path)
            print(f'    ↳ new best acc, saved → {ckpt_path}')

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'    ↳ early stopping at epoch {epoch} (best acc was epoch {best_epoch}).')
                break

    model.load_state_dict(best_state)
    return model

## 10. Run training

In [ ]:
effnet_model = train_model('efficientnet_b0')
print('done training efficientnet_b0')

In [ ]:
import gc
del effnet_model
gc.collect()
torch.cuda.empty_cache()

deit_model = train_model('deit_tiny')
print('done training deit_tiny')

## 11. Verify checkpoints

In [ ]:
for f in sorted(os.listdir(CKPT_DIR)):
    full = os.path.join(CKPT_DIR, f)
    print(f'  {f:40s}  {os.path.getsize(full)/1e6:.1f} MB')

for name in MODELS:
    ckpt = torch.load(os.path.join(CKPT_DIR, f'{name}_best.pt'), map_location='cpu', weights_only=False)
    print(f'\n{name}:  epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}  val_acc={ckpt["val_acc"]:.4f}')

## 12. Export to ONNX

In [ ]:
ONNX_OPSET = 18

def export_to_onnx(name):
    ckpt_path = os.path.join(CKPT_DIR, f'{name}_best.pt')
    onnx_path = os.path.join(CKPT_DIR, f'{name}_best.onnx')
    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model = build_model(name).cpu()
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    torch.onnx.export(
        model, dummy, onnx_path,
        export_params=True,
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
        external_data=False,
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    )
    print(f'  exported → {onnx_path}  ({os.path.getsize(onnx_path)/1e6:.1f} MB)')

for name in MODELS:
    export_to_onnx(name)

## 13. Verify ONNX exports

In [ ]:
import onnxruntime as ort
import numpy as np

for name in MODELS:
    onnx_path = os.path.join(CKPT_DIR, f'{name}_best.onnx')
    sess = ort.InferenceSession(onnx_path)
    dummy = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    out = sess.run(None, {'input': dummy})
    print(f'{name:20s}  output shape: {out[0].shape}   (expect (1, 2))')